# Notebook 04c: Model Comparison and Visualisation

This notebook loads saved experiment results from `results/` and produces all comparison tables and plots. **No training is performed here.** Run notebooks 03, 04, and 04b first to populate the results directory.

Experiments compared:

| ID | Model | Strategy |
|---|---|---|
| dummy | Dummy classifier | Prior frequency |
| cnn | BaselineCNN | Full training from scratch |
| hubert_8b | HuBERT-ECG | Selective unfreezing (last 8 blocks) |
| hubert_lora | HuBERT-ECG | LoRA r=8 |
| hubert_dora | HuBERT-ECG | DoRA r=8 |
| lw | LeadwiseTransformer | Full training from scratch |

**Parameter counts are computed dynamically** by instantiating each model — no hardcoded values.

In [ ]:
import sys, os, json, warnings
sys.path.append('../')
os.environ['TRANSFORMERS_OFFLINE'] = '1'
os.environ['HF_HUB_DISABLE_SYMLINKS_WARNING'] = '1'
warnings.filterwarnings('ignore', category=FutureWarning)
warnings.filterwarnings('ignore', category=UserWarning)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch

from src.utils.config                import CFG
from src.models.hubert_ecg_finetune  import HuBERTECGClassifier, HuBERTECGPEFT
from src.models.leadwise_transformer import LeadwiseTransformer

RESULTS      = CFG['paths']['results']
FIGURES      = CFG['paths']['figures']
SUPERCLASSES = CFG['data']['superclasses']
os.makedirs(FIGURES, exist_ok=True)

print(f'Results dir: {RESULTS}')

## Load Saved Results

Each training run writes a `history.json` with fields `best_auc` and `history` (list of per-epoch dicts). Baseline metrics are stored as flat JSON files. All loading is wrapped in a helper that skips missing files gracefully so partial runs do not break the notebook.

In [ ]:
def load_hist(name):
    path = os.path.join(RESULTS, name, 'history.json')
    if not os.path.exists(path):
        print(f'  [skip] {name}/history.json not found')
        return None
    with open(path) as f:
        return json.load(f)

def load_json(path):
    if not os.path.exists(path):
        print(f'  [skip] {path} not found')
        return None
    with open(path) as f:
        return json.load(f)

# Training histories
hist_8b   = load_hist('hubert_ecg_blocks8')
hist_lora = load_hist('hubert_ecg_lora_r8')
hist_dora = load_hist('hubert_ecg_dora_r8')
hist_lw   = load_hist('leadwise_transformer')

# Flat metrics for baselines
dummy_m = load_json(os.path.join(RESULTS, 'dummy_metrics.json'))
cnn_m   = load_json(os.path.join(RESULTS, 'baseline_cnn', 'baseline_cnn_metrics.json'))

print('\nLoaded:')
for label, h in [('HuBERT 8 blocks', hist_8b), ('HuBERT LoRA r=8', hist_lora),
                 ('HuBERT DoRA r=8', hist_dora), ('Leadwise (full)', hist_lw)]:
    if h:
        print(f'  {label}: best AUC {h["best_auc"]:.4f}  ({len(h["history"])} epochs)')
if dummy_m: print(f'  Dummy: AUC {dummy_m["auc_macro"]:.4f}')
if cnn_m:   print(f'  CNN:   AUC {cnn_m["auc_macro"]:.4f}')

## Compute Parameter Counts

Each model is instantiated once to read its parameter counts from `count_parameters()`. Models are immediately deleted after the count to keep GPU memory free. No hardcoded totals are used — all percentages are computed from the live model.

In [ ]:
print('Instantiating models to count parameters (no training)...')

# HuBERT selective unfreezing (8 blocks)
m_8b  = HuBERTECGClassifier(size='base', blocks_to_unfreeze=8)
p_8b  = m_8b.count_parameters()
del m_8b

# HuBERT LoRA r=8
m_hl  = HuBERTECGPEFT(rank=8, use_dora=False)
p_hl  = m_hl.count_parameters()
del m_hl; torch.cuda.empty_cache()

# HuBERT DoRA r=8
m_hd  = HuBERTECGPEFT(rank=8, use_dora=True)
p_hd  = m_hd.count_parameters()
del m_hd; torch.cuda.empty_cache()

# Leadwise full training
m_lw  = LeadwiseTransformer()
p_lw  = m_lw.count_parameters()
del m_lw

# CNN baseline (approximate — load from checkpoint would give exact count)
from src.models.baseline_cnn import BaselineCNN
m_cnn = BaselineCNN()
p_cnn = sum(p.numel() for p in m_cnn.parameters())
del m_cnn

print('Done.')
print(f'  HuBERT 8b:    {p_8b["trainable"]:>12,} / {p_8b["total"]:,} = {p_8b["percentage"]}')
print(f'  HuBERT LoRA:  {p_hl["trainable"]:>12,} / {p_hl["total"]:,} = {p_hl["percentage"]}')
print(f'  HuBERT DoRA:  {p_hd["trainable"]:>12,} / {p_hd["total"]:,} = {p_hd["percentage"]}')
print(f'  Leadwise:     {p_lw["trainable"]:>12,} / {p_lw["total"]:,} = {p_lw["percentage"]}')
print(f'  CNN:          {p_cnn:>12,} (fully trained)')

## Full Comparison Table

Best macro AUC and macro F1 across all epochs for each experiment. The `vs CNN` column shows the AUC gap relative to the CNN baseline (positive = better than CNN).

In [ ]:
def best_auc(h): return h['best_auc'] if h else float('nan')
def best_f1(h):
    if not h: return float('nan')
    return max(e['f1_macro'] for e in h['history'])

cnn_auc = cnn_m['auc_macro'] if cnn_m else float('nan')

rows = []
if dummy_m:
    rows.append({'Model': 'Dummy',          'Strategy': 'Prior frequency',      'AUC': dummy_m['auc_macro'], 'F1': dummy_m['f1_macro'],   'Trainable': 0,                  'Total': 0})
if cnn_m:
    rows.append({'Model': 'CNN',            'Strategy': 'Full training',        'AUC': cnn_auc,              'F1': cnn_m['f1_macro'],     'Trainable': p_cnn,              'Total': p_cnn})
if hist_8b:
    rows.append({'Model': 'HuBERT-ECG',    'Strategy': 'Selective 8 blocks',   'AUC': best_auc(hist_8b),    'F1': best_f1(hist_8b),      'Trainable': p_8b['trainable'],  'Total': p_8b['total']})
if hist_lora:
    rows.append({'Model': 'HuBERT-ECG',    'Strategy': 'LoRA r=8',             'AUC': best_auc(hist_lora),  'F1': best_f1(hist_lora),    'Trainable': p_hl['trainable'],  'Total': p_hl['total']})
if hist_dora:
    rows.append({'Model': 'HuBERT-ECG',    'Strategy': 'DoRA r=8',             'AUC': best_auc(hist_dora),  'F1': best_f1(hist_dora),    'Trainable': p_hd['trainable'],  'Total': p_hd['total']})
if hist_lw:
    rows.append({'Model': 'LeadwiseTF',    'Strategy': 'Full training',        'AUC': best_auc(hist_lw),    'F1': best_f1(hist_lw),      'Trainable': p_lw['trainable'],  'Total': p_lw['total']})

df = pd.DataFrame(rows)
df['Params%']  = df.apply(lambda r: f"{100*r['Trainable']/r['Total']:.1f}%" if r['Total'] > 0 else 'n/a', axis=1)
df['vs CNN']   = (df['AUC'] - cnn_auc).map(lambda x: f'+{x:.4f}' if x > 0 else f'{x:.4f}')

display_df = df[['Model', 'Strategy', 'AUC', 'F1', 'Params%', 'vs CNN']]
print(display_df.to_string(index=False))

## Learning Curves

Validation macro AUC over training epochs for all four trained models. Horizontal dashed lines mark the CNN and dummy baselines.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))

palette = [
    ('HuBERT 8 blocks',  hist_8b,   'tab:blue',   '-'),
    ('HuBERT LoRA r=8',  hist_lora, 'tab:orange',  '-'),
    ('HuBERT DoRA r=8',  hist_dora, 'tab:purple',  '-'),
    ('Leadwise (full)',   hist_lw,   'tab:green',   '-'),
]

for label, h, color, ls in palette:
    if not h:
        continue
    epochs = [e['epoch']     for e in h['history']]
    aucs   = [e['auc_macro'] for e in h['history']]
    ax.plot(epochs, aucs, label=label, color=color, linestyle=ls, linewidth=2, marker='o', markersize=4)

if cnn_m:
    ax.axhline(y=cnn_auc, color='red',  linestyle='--', linewidth=1.5, label=f'CNN baseline ({cnn_auc:.4f})')
if dummy_m:
    ax.axhline(y=dummy_m['auc_macro'], color='gray', linestyle=':', linewidth=1.2, label='Dummy baseline (0.50)')

ax.set_title('Validation AUC over training epochs')
ax.set_xlabel('Epoch')
ax.set_ylabel('Macro AUC')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(FIGURES, 'learning_curves.png'), dpi=150)
plt.show()

## Per-Class AUC

Grouped bar chart of per-class AUC at the best epoch (highest macro AUC) for each experiment. The five PTB-XL superclasses differ substantially in prevalence: `NORM` (44.5%) is the majority class, while `HYP` (12.4%) is the rarest and typically hardest to classify.

In [ ]:
def best_epoch(h):
    """Return the epoch dict with the highest auc_macro."""
    return max(h['history'], key=lambda e: e['auc_macro'])

named = []
for label, h in [('HuBERT 8b', hist_8b), ('HuBERT LoRA', hist_lora),
                 ('HuBERT DoRA', hist_dora), ('Leadwise', hist_lw)]:
    if h:
        named.append((label, best_epoch(h)['per_class']))

n_exp = len(named)
n_cls = len(SUPERCLASSES)
x     = np.arange(n_cls)
width = 0.7 / max(n_exp, 1)

fig, ax = plt.subplots(figsize=(11, 5))
colors  = ['tab:blue', 'tab:orange', 'tab:purple', 'tab:green']

for i, (name, per_class) in enumerate(named):
    vals   = [per_class.get(cls, 0.0) for cls in SUPERCLASSES]
    offset = (i - (n_exp - 1) / 2) * width
    ax.bar(x + offset, vals, width, label=name, color=colors[i % len(colors)])

ax.axhline(y=0.5, color='gray', linestyle=':', linewidth=1, label='Chance')
ax.set_xticks(x)
ax.set_xticklabels(SUPERCLASSES, fontsize=11)
ax.set_ylim(0, 1.05)
ax.set_title('Per-class AUC at best epoch')
ax.set_xlabel('Diagnostic superclass')
ax.set_ylabel('AUC')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.savefig(os.path.join(FIGURES, 'per_class_auc.png'), dpi=150)
plt.show()

## Parameter Efficiency Table

Trainable parameters and their share of total model parameters for each method.

In [ ]:
print(f"{'Method':<28s}  {'Trainable':>14s}  {'Total':>14s}  {'% trainable':>12s}")
print('-' * 74)

rows_p = [
    ('Dummy',               0,                   0,                    'n/a'),
    ('CNN (baseline)',       p_cnn,               p_cnn,                '100.0%'),
    ('HuBERT selective 8b', p_8b['trainable'],   p_8b['total'],        p_8b['percentage']),
    ('HuBERT LoRA r=8',     p_hl['trainable'],   p_hl['total'],        p_hl['percentage']),
    ('HuBERT DoRA r=8',     p_hd['trainable'],   p_hd['total'],        p_hd['percentage']),
    ('Leadwise full train',  p_lw['trainable'],   p_lw['total'],        p_lw['percentage']),
]
for name, trainable, total, pct in rows_p:
    t_str  = f'{trainable:,}' if trainable > 0 else 'n/a'
    tot_str = f'{total:,}'    if total     > 0 else 'n/a'
    print(f'{name:<28s}  {t_str:>14s}  {tot_str:>14s}  {pct:>12s}')

## Parameter Efficiency vs. Performance Scatter

Each point is one experiment. X-axis: number of trainable parameters (log scale). Y-axis: best macro AUC. A model in the top-left corner is most efficient — high accuracy at low training cost. The CNN baseline is plotted as a dashed horizontal reference.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))

# (label, trainable_params, auc, color, marker)
points = []
if cnn_m:     points.append(('CNN',              p_cnn,               cnn_auc,                   'red',        'o'))
if hist_8b:   points.append(('HuBERT 8 blocks',  p_8b['trainable'],   best_auc(hist_8b),          'tab:blue',   's'))
if hist_lora: points.append(('HuBERT LoRA r=8',  p_hl['trainable'],   best_auc(hist_lora),        'tab:orange', '^'))
if hist_dora: points.append(('HuBERT DoRA r=8',  p_hd['trainable'],   best_auc(hist_dora),        'tab:purple', 'D'))
if hist_lw:   points.append(('Leadwise (full)',   p_lw['trainable'],   best_auc(hist_lw),          'tab:green',  '^'))

for label, params, auc, color, marker in points:
    ax.scatter(params, auc, color=color, marker=marker, s=130, zorder=3)
    ax.annotate(label, (params, auc), textcoords='offset points',
                xytext=(8, 4), fontsize=9, color=color)

if cnn_m:
    ax.axhline(y=cnn_auc, color='red', linestyle=':', linewidth=1, alpha=0.5, label='CNN baseline')
if dummy_m:
    ax.axhline(y=dummy_m['auc_macro'], color='gray', linestyle=':', linewidth=1, alpha=0.4, label='Dummy')

ax.set_xscale('log')
ax.set_xlabel('Trainable parameters (log scale)', fontsize=11)
ax.set_ylabel('Best macro AUC', fontsize=11)
ax.set_title('Parameter Efficiency vs. Performance', fontsize=13)
ax.grid(True, alpha=0.3)
ax.legend(fontsize=9)
plt.tight_layout()
plt.savefig(os.path.join(FIGURES, 'efficiency_scatter.png'), dpi=150)
plt.show()

## Efficiency Profiling

Wall-clock training time, peak GPU memory, and checkpoint size for each experiment. Profiling data is loaded from `profiling.json` files written alongside `history.json` by each training run.

Experiments without a `profiling.json` (e.g., trained before this feature was added) are skipped with a warning.

In [ ]:
from src.utils.profiler import ExperimentProfiler

EXPERIMENTS = [
    "dummy_classifier",
    "baseline_cnn",
    "hubert_ecg_blocks8",
    "hubert_ecg_lora_r8",
    "hubert_ecg_dora_r8",
    "leadwise_transformer",
]

profiling = {}
for exp in EXPERIMENTS:
    path = os.path.join(RESULTS, exp)
    p = ExperimentProfiler.load(path)
    if p is not None:
        profiling[exp] = p
    else:
        print(f"Warning: no profiling.json found for {exp}")

rows = []
for exp in EXPERIMENTS:
    p = profiling.get(exp)
    if p is None:
        continue
    rows.append({
        "Experiment":     exp,
        "Total time":     p["total_time_human"],
        "Avg epoch (s)":  p["avg_epoch_time_sec"],
        "Peak VRAM (GB)": p["peak_gpu_memory_gb"],
        "Trainable %":    f"{p['trainable_pct']}%",
        "Trainable M":    round(p["trainable_params"] / 1e6, 2),
        "Total M":        round(p["total_params"] / 1e6, 2),
        "Checkpoint MB":  p["checkpoint_size_mb"],
    })

df_efficiency = pd.DataFrame(rows)
print("\n" + "=" * 80)
print("EFFICIENCY COMPARISON -- all experiments")
print("=" * 80)
print(df_efficiency.to_string(index=False))

csv_path = os.path.join(RESULTS, 'efficiency_comparison.csv')
df_efficiency.to_csv(csv_path, index=False)
print(f"\nSaved -> {csv_path}")

In [ ]:
# -- Inference: load all models and run predict_proba on val set --------------
# Models are loaded and deleted one at a time to stay within 8 GB VRAM.
# Each call to predict_proba returns (N_records, 5) probs and labels.

import torch
from src.utils.config               import CFG
from src.preprocessing.label_utils  import load_all_labels
from src.preprocessing.dataset      import ECGDataset
from src.preprocessing.dataset_full import ECGDatasetFull
from src.inference.loaders          import (load_baseline_cnn, load_hubert_blocks,
                                             load_hubert_peft, load_leadwise)
from src.inference.predict          import predict_proba

DATA_PATH = CFG['data']['path']
device    = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

Y      = load_all_labels(DATA_PATH + 'ptbxl_database.csv', DATA_PATH + 'scp_statements.csv')
val_df = Y[Y.strat_fold == 9]

# BaselineCNN
print('Loading BaselineCNN...')
_m, _ds = load_baseline_cnn()
cnn_probs, cnn_labels = predict_proba(_m.to(device), _ds(val_df, DATA_PATH), device)
del _m; torch.cuda.empty_cache()
print(f'  CNN done -- {cnn_probs.shape[0]} records')

# HuBERT selective 8 blocks
print('Loading HuBERT 8 blocks...')
_m, _ds = load_hubert_blocks(n=8)
h8b_probs, h8b_labels = predict_proba(_m.to(device), _ds(val_df, DATA_PATH), device)
del _m; torch.cuda.empty_cache()
print(f'  HuBERT 8b done -- {h8b_probs.shape[0]} records')

# HuBERT LoRA r=8
print('Loading HuBERT LoRA r=8...')
_m, _ds = load_hubert_peft(rank=8, use_dora=False)
hl_probs, hl_labels = predict_proba(_m.to(device), _ds(val_df, DATA_PATH), device)
del _m; torch.cuda.empty_cache()
print(f'  HuBERT LoRA done -- {hl_probs.shape[0]} records')

# HuBERT DoRA r=8
print('Loading HuBERT DoRA r=8...')
_m, _ds = load_hubert_peft(rank=8, use_dora=True)
hd_probs, hd_labels = predict_proba(_m.to(device), _ds(val_df, DATA_PATH), device)
del _m; torch.cuda.empty_cache()
print(f'  HuBERT DoRA done -- {hd_probs.shape[0]} records')

# LeadwiseTransformer
print('Loading LeadwiseTransformer...')
_m, _ds = load_leadwise()
lw_probs, lw_labels = predict_proba(_m.to(device), _ds(val_df, DATA_PATH), device)
del _m; torch.cuda.empty_cache()
print(f'  Leadwise done -- {lw_probs.shape[0]} records')

In [ ]:
# -- Per-class Precision / Recall / F1 table ----------------------------------
import pandas as pd
from sklearn.metrics import precision_score, recall_score, f1_score

THRESHOLD    = 0.5
SUPERCLASSES = CFG['data']['superclasses']

model_preds = [
    ('CNN',         cnn_probs, cnn_labels),
    ('HuBERT 8b',   h8b_probs, h8b_labels),
    ('HuBERT LoRA', hl_probs,  hl_labels),
    ('HuBERT DoRA', hd_probs,  hd_labels),
    ('Leadwise',    lw_probs,  lw_labels),
]

rows = []
for name, probs, labels in model_preds:
    preds = (probs.numpy() >= THRESHOLD).astype(int)
    lbls  = labels.numpy().astype(int)
    row   = {'Model': name}
    for i, cls in enumerate(SUPERCLASSES):
        row[f'{cls}_P']  = round(precision_score(lbls[:, i], preds[:, i], zero_division=0), 3)
        row[f'{cls}_R']  = round(recall_score(   lbls[:, i], preds[:, i], zero_division=0), 3)
        row[f'{cls}_F1'] = round(f1_score(       lbls[:, i], preds[:, i], zero_division=0), 3)
    row['MacroP']  = round(precision_score(lbls, preds, average='macro', zero_division=0), 3)
    row['MacroR']  = round(recall_score(   lbls, preds, average='macro', zero_division=0), 3)
    row['MacroF1'] = round(f1_score(       lbls, preds, average='macro', zero_division=0), 3)
    rows.append(row)

df_pr = pd.DataFrame(rows).set_index('Model')
print(df_pr.to_string())

In [ ]:
# -- Per-class Precision-Recall curves ----------------------------------------
import matplotlib.pyplot as plt
from sklearn.metrics import precision_recall_curve, average_precision_score

fig, axes = plt.subplots(1, 5, figsize=(18, 4))

palette = [
    ('CNN',         cnn_probs, cnn_labels, 'red'),
    ('HuBERT 8b',   h8b_probs, h8b_labels, 'tab:blue'),
    ('HuBERT LoRA', hl_probs,  hl_labels,  'tab:orange'),
    ('HuBERT DoRA', hd_probs,  hd_labels,  'tab:purple'),
    ('Leadwise',    lw_probs,  lw_labels,  'tab:green'),
]

for col, cls in enumerate(SUPERCLASSES):
    ax = axes[col]
    for name, probs, labels, color in palette:
        p_arr, r_arr, _ = precision_recall_curve(
            labels.numpy()[:, col], probs.numpy()[:, col]
        )
        ap = average_precision_score(labels.numpy()[:, col], probs.numpy()[:, col])
        ax.plot(r_arr, p_arr, label=f'{name} (AP={ap:.2f})', color=color, linewidth=1.5)
    ax.set_title(cls, fontsize=11)
    ax.set_xlabel('Recall')
    ax.set_ylabel('Precision')
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1.05)
    ax.legend(fontsize=7)
    ax.grid(True, alpha=0.3)

plt.suptitle('Precision-Recall Curves per Class', fontsize=13)
plt.tight_layout()
plt.savefig(os.path.join(FIGURES, 'pr_curves.png'), dpi=150)
plt.show()

In [ ]:
# -- Best transformer ---------------------------------------------------------
# hist_8b, hist_lora, hist_dora, hist_lw are loaded earlier in this notebook.
transformer_histories = {
    'HuBERT selective 8b': hist_8b,
    'HuBERT LoRA r=8':     hist_lora,
    'HuBERT DoRA r=8':     hist_dora,
    'Leadwise (full)':     hist_lw,
}

best_name = max(
    (k for k, v in transformer_histories.items() if v),
    key=lambda k: transformer_histories[k]['best_auc']
)
best_val = transformer_histories[best_name]['best_auc']

print(f'Best transformer: {best_name}  |  AUC {best_val:.4f}')
print(f'(Use this model in 06_clinical_demo.ipynb)')